<a href="https://colab.research.google.com/github/sudoice/btpii/blob/main/main_experiments.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np

# TD3 Hyperparameters
TD3_CONFIG = {
    'actor_lr': 0.0001,              # Learning rate for actor
    'critic_lr': 0.001,             # Learning rate for critic
    'discount_factor': 0.99,        # Discount factor ζ
    'tau': 0.005,                   # Soft update rate ψ
    'policy_noise': 0.1,            # Policy noise for target smoothing
    'noise_clip': 0.5,              # Noise clip value c
    'policy_delay': 2,              # Delayed policy updates k
    'batch_size': 256,              # Mini-batch size
    'buffer_size': 100000,         # Replay buffer size
    'hidden_layers': [800, 600],    # Hidden layer neurons
    'max_episodes': 5000,           # Maximum training episodes
    'max_steps': 100,               # Maximum steps per episode
}

# VEC Network Parameters
NETWORK_CONFIG = {
    # Simulation parameters
    'time_slot_duration': 0.05,     # τ = 0.05 seconds
    'carrier_frequency': 1.95e9,       # f = 1.95 GHz
    'speed_of_light': 3e8,          # C = 3 × 10^8 m/s

    # UAV parameters
    'uav_height': 100,              # H = 100 m
    'uav_max_velocity': 50,         # v_max = 50 m/s
    'uav_max_energy': 3000,         # E_max_U = 3000 J
    'uav_compute_capacity': 5e9,   # F_U = 5 GHz
    'uav_bandwidth': 50e6,           # W_U = 50 MHz
    'uav_max_associations': 5,      # N_max_U = 5
    'uav_rental_price': 5e-8,        # P_U = 0.5 pence/CPU cycle

    # RSU parameters
    'rsu_compute_capacity': 2e9,   # F_R = 2 GHz
    'rsu_bandwidth': 20e6,           # W_R = 20 MHz
    'rsu_max_associations': 5,      # N_max_M = 5
    'rsu_rental_price': 3e-8,        # P_R = 0.3 pence/CPU cycle
    'rsu_coverage_radius': 100,     # 100 m

    # Vehicle parameters
    'vehicle_compute_capacity': 0.5e9,    # F_n = 1 GHz
    'vehicle_transmit_power': 10e-3,      # p_n = 0.5 W
    'vehicle_velocity_range': [10, 20], # 10-20 m/s

    # Channel parameters
    'awgn_power_density': 3.98e-21,     # N_0 = -174 dBm/Hz
    'eta_los': 1,                   # LoS path loss (dB)
    'eta_nlos': 20,                 # NLoS path loss (dB)
    'eta_rayleigh': 0.707,              # Rayleigh fading coefficient
    'a_param': 10,
    'b_param': 0.4,               # Environment parameter b

    # Computation parameters
    'cpu_cycles_per_bit': 1000,     # X_c = 1000 cycles/bit
    'kappa_vehicle': 1e-27,
    'kappa_uav': 1e-27,            # k_u = 10^-28

    # Task parameters
    'task_data_size_mean': 500e3,   # Mean task size: 500 kB (in bits)
    'task_data_size_range': [500*8*1024, 800*8*1024],
    'task_data_size_std': 100e3,    # Std of task size: 100 kB
    'task_max_latency': 0.5,        # T_max = 1 second
    'task_arrival_rate': 0.6,       # λ_u = 0.6 (Poisson parameter)

    # NOMA & Power Allocation parameters
    'vehicle_max_power': 20e-3,      # P_max = 20 mW (replacing static transmit power)

    # Reliability (PER) parameters
    'per_a': 1.0,                    # Constant 'a' for PER exponential bound
    'per_b': 0.001,                    # Constant 'b' for PER exponential bound

    # UAV energy model parameters (from equation 20)
    'P0': 79.8563,                  # Blade profile power (W)
    'Pi': 88.6279,                  # Induced power in hovering (W)
    'Utip': 120,                    # Tip speed of rotor blade (m/s)
    'v0': 4.03,                     # Mean rotor induced velocity (m/s)
    'd0': 0.6,                      # Fuselage drag ratio
    'rho0': 1.225,                  # Air density (kg/m^3)
    's0': 0.05,                     # Rotor solidity
    'A0': 0.503,                    # Rotor disc area (m^2)

    # Cost weights (Make sure they sum to 1.0 in your scenarios)
    'gamma_A': 0.25,                 # Adjusted weight for AoI
    'gamma_E': 0.25,                 # Adjusted weight for energy
    'gamma_P': 0.25,                 # Adjusted weight for rental price
    'gamma_R': 0.25,                 # NEW: Weight for Reliability (PER) penalty

    # Penalty constants
    'c1': 2,                     # Penalty for association constraint violation
    'c2': 2,                     # Penalty for latency constraint violation
    'c3': 2,                     # Penalty for energy constraint violation
    'topsis_enabled': True,
    'load_threshold_ratio': 1.5,
    'topsis_w_resource': 0.6,
    'topsis_w_distance': 0.4,
}

# Map Configuration
MAP_CONFIGS = {
    'map1': {
        'description': '600m × 400m with 3 RSUs',
        'area': (600, 400),
        'rsu_positions': [
            (100, 100),
            (300, 300),
            (500, 100),
        ],
        'num_rsus': 3,
    },
    'map2': {
        'description': '1000m x 300m highway — 3 RSUs spread far apart',
        'area': (1000, 300),
        'rsu_positions': [
            (100, 150),
            (500, 150),
            (900, 150),
        ],
        'num_rsus': 3,
    }
}

# Scenario Configurations (Updated for 4-way optimization)
SCENARIO_CONFIGS = {
    'normal': {
        'gamma_A': 0.25,
        'gamma_E': 0.25,
        'gamma_P': 0.25,
        'gamma_R': 0.25,
    },
    'congested': {
        'gamma_A': 0.30,
        'gamma_E': 0.25,
        'gamma_P': 0.20,
        'gamma_R': 0.25,
        'rsu_compute_capacity': 1e9,    # 1 GHz
        'rsu_max_associations': 3,      # 3
    }
}

def get_config(scenario='normal', map_name='map1', num_vehicles=80):
    config = {
        'td3': TD3_CONFIG.copy(),
        'network': NETWORK_CONFIG.copy(),
        'map': MAP_CONFIGS[map_name].copy(),
        'num_vehicles': num_vehicles,
    }
    if scenario in SCENARIO_CONFIGS:
        config['network'].update(SCENARIO_CONFIGS[scenario])
    return config

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_SAVE_DIR = '/content/drive/MyDrive/RESULTS'

In [ ]:
"""
UAV-Assisted VEC Environment
"""

import numpy as np
import gym
from gym import spaces


class UAVAssistedVECEnv(gym.Env):
    """
    Environment for UAV-Assisted Vehicular Edge Computing
    """

    def __init__(self, config):
        super(UAVAssistedVECEnv, self).__init__()

        self.config = config
        self.is_noma = config.get('is_noma', False)
        self.use_topsis = config.get('use_topsis', False)
        self.td3_cfg = config['td3']
        self.net_cfg = config['network']
        self.map_cfg = config['map']
        self.num_vehicles = config['num_vehicles']

        # Extract key parameters
        self.tau = self.net_cfg['time_slot_duration']
        self.num_rsus = self.map_cfg['num_rsus']
        self.num_ens = self.num_rsus + 1  # RSUs + UAV

        # Initialize RSU positions
        self.rsu_positions = np.array(self.map_cfg['rsu_positions'])

        # Define action and observation spaces
        self._setup_spaces()

        # Initialize state variables
        self.reset()

    def _setup_spaces(self):
        """Setup action and observation spaces"""
        # Base Action space: [v_u, theta_u, alpha_1...alpha_N]
        action_low = [0, 0] + [0] * self.num_vehicles
        action_high = [self.net_cfg['uav_max_velocity'], 2 * np.pi] + [1] * self.num_vehicles

        # If NOMA is enabled, expand action space for power allocations
        if self.net_cfg.get('noma_enabled', False):
            action_low += [0] * self.num_vehicles
            action_high += [self.net_cfg['vehicle_max_power']] * self.num_vehicles

        self.action_space = spaces.Box(
            low=np.array(action_low),
            high=np.array(action_high),
            dtype=np.float32
        )

        # [State space setup remains exactly the same below this...]

        # State space dimension calculation (from equation 29)
        # W(t): 2(N+1) - coordinates of vehicles and UAV
        # U(t): 2N - task information (D_n, T_max_n)
        # Y(t): N - time intervals
        # Q(t): N + M + 1 - queue backlogs
        # H(t): N(M+1) - channel gains
        # E_res_u: 1 - residual UAV energy
        state_dim = (
            2 * (self.num_vehicles + 1) +  # W(t)
            2 * self.num_vehicles +          # U(t)
            self.num_vehicles +              # Y(t)
            self.num_vehicles + self.num_ens +  # Q(t)
            self.num_vehicles * self.num_ens +  # H(t)
            1                                # E_res_u
        )

        self.observation_space = spaces.Box(
            low=-np.inf,
            high=np.inf,
            shape=(state_dim,),
            dtype=np.float32
        )

    def reset(self):
        """Reset environment to initial state"""
        area_width, area_height = self.map_cfg['area']

        # --- YOUR NEW CONGESTION LOGIC ---
        # If we are using map4, force vehicles into clusters to test TOPSIS
        if self.num_rsus == 3 and area_width == 1000: # Simple check for map4
            self.vehicle_positions = np.zeros((self.num_vehicles, 2))
            n_half = self.num_vehicles // 2
            rsu0 = np.array(self.rsu_positions[0], dtype=float)
            rsu2 = np.array(self.rsu_positions[min(2, len(self.rsu_positions)-1)], dtype=float)

            self.vehicle_positions[:n_half] = np.random.normal(
                loc=rsu0, scale=[30, 25], size=(n_half, 2))
            self.vehicle_positions[n_half:] = np.random.normal(
                loc=rsu2, scale=[30, 25], size=(self.num_vehicles - n_half, 2))

            self.vehicle_positions[:, 0] = np.clip(self.vehicle_positions[:, 0], 0, area_width)
            self.vehicle_positions[:, 1] = np.clip(self.vehicle_positions[:, 1], 0, area_height)
        else:
            # Standard uniform distribution for maps 1, 2, and 3
            self.vehicle_positions = np.random.uniform(
                low=[0, 0], high=[area_width, area_height], size=(self.num_vehicles, 2)
            )

        # Initialize vehicle velocities and angles
        v_min, v_max = self.net_cfg['vehicle_velocity_range']
        self.vehicle_velocities = np.random.uniform(v_min, v_max, self.num_vehicles)
        self.vehicle_angles = np.random.uniform(0, 2*np.pi, self.num_vehicles)

        # Initialize UAV
        self.uav_position = np.array([area_width/2, area_height/2])
        self.uav_height = self.net_cfg['uav_height']
        self.uav_residual_energy = self.net_cfg['uav_max_energy']

        # Start queues at zero
        self.vehicle_queues = np.zeros(self.num_vehicles)
        self.en_queues = np.zeros(self.num_ens)

        self.task_intervals = np.zeros(self.num_vehicles)
        self.current_tasks, arrivals = self._generate_tasks()
        self.aoi = np.zeros(self.num_vehicles)
        self.current_step = 0

        return self._get_state()

    def _normalize_state(self, state):
        """
        Normalize state features to roughly [0,1] range.
        This improves TD3 stability without changing the MDP.
        """

        state = state.copy()

        idx = 0

        area_width, area_height = self.map_cfg['area']


        # W(t): positions  → normalize by map size
        # size = 2*(N+1)
        num_pos = 2 * (self.num_vehicles + 1)
        for i in range(0, num_pos, 2):
            state[idx + i] /= area_width
            state[idx + i + 1] /= area_height
        idx += num_pos

        # U(t): task info (D_n, T_max_n)
        # normalize by configured maxima
        # size = 2N
        D_range = self.net_cfg['task_data_size_range']
        D_max = max(D_range)
        T_max_global = self.net_cfg['task_max_latency']

        for i in range(self.num_vehicles):
            state[idx + 2*i] /= D_max
            state[idx + 2*i + 1] /= T_max_global
        idx += 2 * self.num_vehicles

        # Y(t): inter-arrival times
        # normalize by a safe horizon (seconds)
        # size = N
        interval_norm = 5.0  # safe default horizon
        state[idx: idx + self.num_vehicles] /= interval_norm
        idx += self.num_vehicles

        # Q(t): queues
        # size = N + M + 1
        Q_max = self.net_cfg.get('queue_capacity', 1e8)

        total_q = self.num_vehicles + self.num_ens
        state[idx: idx + total_q] /= Q_max
        idx += total_q

        # H(t): channel gains
        # already small → just clip
        # size = N*(M+1)
        num_h = self.num_vehicles * self.num_ens
        state[idx: idx + num_h] = np.clip(state[idx: idx + num_h], 0.0, 1.0)
        idx += num_h

        # UAV residual energy
        state[idx] /= self.net_cfg['uav_max_energy']

        return state.astype(np.float32)

    def _generate_tasks(self):
        tasks = []
        arrivals = np.zeros(self.num_vehicles)

        lambda_u = self.net_cfg['task_arrival_rate']
        tau = self.net_cfg['time_slot_duration']
        p_arrival = 1 - np.exp(-lambda_u * tau)

        for n in range(self.num_vehicles):
            if np.random.random() < p_arrival:
                data_size_range = self.net_cfg['task_data_size_range']
                data_size = np.random.uniform(
                    data_size_range[0],
                    data_size_range[1]
                )
                max_latency = self.net_cfg['task_max_latency']

                tasks.append((data_size, max_latency))
                arrivals[n] = 1
            else:
                tasks.append(None)

        return tasks, arrivals

    def _compute_path_loss_a2g(self, vehicle_idx):
        """
        Compute Air-to-Ground path loss

        Args:
            vehicle_idx: Index of the vehicle

        Returns:
            Path loss in dB
        """
        # Distance between UAV and vehicle
        vehicle_pos = self.vehicle_positions[vehicle_idx]
        uav_pos_3d = np.array([self.uav_position[0], self.uav_position[1], self.uav_height])
        vehicle_pos_3d = np.array([vehicle_pos[0], vehicle_pos[1], 0])

        distance = np.linalg.norm(uav_pos_3d - vehicle_pos_3d)

        # Elevation angle
        theta_elevation = np.arcsin(self.uav_height / distance) if distance > 0 else 0

        # LoS probability
        a = self.net_cfg['a_param']
        b = self.net_cfg['b_param']
        theta_deg = np.degrees(theta_elevation)
        p_los = 1 / (1 + a * np.exp(-b * (theta_deg - a)))

        # Path loss
        f = self.net_cfg['carrier_frequency']
        c = self.net_cfg['speed_of_light']
        eta_los = self.net_cfg['eta_los']
        eta_nlos = self.net_cfg['eta_nlos']

        fspl = 20 * np.log10(distance * 4 * np.pi * f / c)
        path_loss = fspl + p_los * eta_los + (1 - p_los) * eta_nlos

        return path_loss

    def _compute_path_loss_g2g(self, vehicle_idx, rsu_idx):
        """
        Compute Ground-to-Ground path loss

        Args:
            vehicle_idx: Index of the vehicle
            rsu_idx: Index of the RSU

        Returns:
            Path loss in dB
        """
        vehicle_pos = self.vehicle_positions[vehicle_idx]
        rsu_pos = self.rsu_positions[rsu_idx]

        distance = np.linalg.norm(vehicle_pos - rsu_pos)

        f = self.net_cfg['carrier_frequency']
        c = self.net_cfg['speed_of_light']
        eta_rayleigh = self.net_cfg['eta_rayleigh']

        fspl = 20 * np.log10(distance * 4 * np.pi * f / c) if distance > 0 else 0
        path_loss = fspl + eta_rayleigh

        return path_loss

    def _compute_channel_gains(self):
        """
        Compute channel gains h_n,m for all vehicle-EN pairs

        Returns:
            Channel gains matrix of shape (num_vehicles, num_ens)
        """
        channel_gains = np.zeros((self.num_vehicles, self.num_ens))

        for n in range(self.num_vehicles):
            # Channel gains to RSUs
            for m in range(self.num_rsus):
                path_loss_db = self._compute_path_loss_g2g(n, m)
                channel_gains[n, m] = 10 ** (-path_loss_db / 10)

            # Channel gain to UAV
            path_loss_db = self._compute_path_loss_a2g(n)
            channel_gains[n, self.num_rsus] = 10 ** (-path_loss_db / 10)

        return channel_gains

    def _compute_noma_rates_and_per(self, associations, allocated_powers):
        channel_gains = self._compute_channel_gains()
        rates = np.zeros(self.num_vehicles)
        pers = np.zeros(self.num_vehicles)
        sic_violations = 0  # NEW: Track illegal power allocations

        N0 = self.net_cfg['awgn_power_density']
        a_param = self.net_cfg['per_a']
        b_param = self.net_cfg['per_b']

        for m in range(self.num_ens):
            assoc_indices = np.where(associations[:, m] == 1)[0]
            if len(assoc_indices) == 0: continue

            gains_m = channel_gains[assoc_indices, m]
            powers_m = allocated_powers[assoc_indices]

            # Sort channels strongest to weakest
            sort_idx = np.argsort(gains_m)[::-1]
            sorted_vehicles = assoc_indices[sort_idx]
            sorted_gains = gains_m[sort_idx]

            sorted_powers = np.sort(powers_m)

            bandwidth = self.net_cfg['rsu_bandwidth'] if m < self.num_rsus else self.net_cfg['uav_bandwidth']
            noise_power = N0 * bandwidth + 1e-12

            for i in range(len(sorted_vehicles)):
                vehicle_n = sorted_vehicles[i]
                p_n = sorted_powers[i]
                h_nm = sorted_gains[i]

                interference = 0.0
                for j in range(i + 1, len(sorted_vehicles)):
                    interference += sorted_powers[j] * sorted_gains[j]

                sinr = (p_n * h_nm) / (interference + noise_power)
                rates[vehicle_n] = bandwidth * np.log2(1 + sinr)
                pers[vehicle_n] = a_param * np.exp(-b_param * sinr)

        return rates, pers, 0

    def _compute_transmission_rate(self, vehicle_idx, en_idx, num_associated_vehicles, h_nm):
        if en_idx < self.num_rsus:
            total_bandwidth = self.net_cfg['rsu_bandwidth']
            max_assoc = self.net_cfg['rsu_max_associations']
        else:
            total_bandwidth = self.net_cfg['uav_bandwidth']
            max_assoc = self.net_cfg['uav_max_associations']

        p_n = self.net_cfg['vehicle_transmit_power']

        N0 = self.net_cfg['awgn_power_density']
        a_param = self.net_cfg['per_a']
        b_param = self.net_cfg['per_b']

        if num_associated_vehicles > max_assoc:
            rate = 1e-6
            per = 1.0
        else:
            beta = 1.0 / num_associated_vehicles if num_associated_vehicles > 0 else 0
            allocated_bandwidth = beta * total_bandwidth
            noise_power = beta * N0 * total_bandwidth + 1e-12

            sinr = p_n * h_nm / noise_power if noise_power > 0 else 0
            rate = allocated_bandwidth * np.log2(1 + sinr)

            per = a_param * np.exp(-b_param * sinr)
            per = min(per, 1.0)

        return rate, per

    def _compute_local_latency(self, vehicle_idx, offload_ratio):
        """
        Compute local computation latency (equations 10-12)

        Args:
            vehicle_idx: Index of the vehicle
            offload_ratio: Offloading ratio o_n

        Returns:
            Local service latency
        """
        task = self.current_tasks[vehicle_idx]
        if task is None:
            return 0

        data_size, _ = task
        local_portion = (1 - offload_ratio) * data_size

        # Queue delay
        F_n = self.net_cfg['vehicle_compute_capacity']
        X_c = self.net_cfg['cpu_cycles_per_bit']
        Q_n = self.vehicle_queues[vehicle_idx]

        tau_queue = X_c * Q_n / F_n

        # Computation delay
        tau_comp = (1 - offload_ratio) * X_c * data_size / F_n

        return tau_queue + tau_comp

    def _compute_offload_latency(self, vehicle_idx, en_idx, offload_ratio, num_associated):
        """
        Compute offloading latency

        Args:
            vehicle_idx: Index of the vehicle
            en_idx: Index of the edge node
            offload_ratio: Offloading ratio o_n
            num_associated: Number of vehicles associated with this EN

        Returns:
            Offloading service latency
        """
        task = self.current_tasks[vehicle_idx]
        if task is None or offload_ratio == 0:
            return 0

        data_size, _ = task
        offload_portion = offload_ratio * data_size

        # Transmission delay
        rate = self._compute_transmission_rate(vehicle_idx, en_idx, num_associated)

        rate = max(rate, 1e-6)
        tau_trans = offload_portion / rate
        tau_trans = min(tau_trans, 10.0)

        # Queue delay at EN
        if en_idx < self.num_rsus:
            F_m = self.net_cfg['rsu_compute_capacity']
        else:
            F_m = self.net_cfg['uav_compute_capacity']

        X_c = self.net_cfg['cpu_cycles_per_bit']
        Q_m = self.en_queues[en_idx]

        tau_queue = X_c * Q_m / F_m

        # Computation delay at EN
        tau_comp = offload_ratio * X_c * data_size / F_m

        return tau_trans + tau_queue + tau_comp

    def _compute_local_energy(self, vehicle_idx, offload_ratio):
        """
        Compute local computation energy

        Args:
            vehicle_idx: Index of the vehicle
            offload_ratio: Offloading ratio o_n

        Returns:
            Local computation energy consumption in Joules
        """
        task = self.current_tasks[vehicle_idx]
        if task is None:
            return 0

        data_size, _ = task
        F_n = self.net_cfg['vehicle_compute_capacity']
        X_c = self.net_cfg['cpu_cycles_per_bit']
        kappa = self.net_cfg['kappa_vehicle']

        energy = kappa * (F_n ** 2) * (1 - offload_ratio) * X_c * data_size

        return energy

    def _compute_offload_energy(self, vehicle_idx, en_idx, offload_ratio, num_associated):
        """
        Compute offloading energy

        Args:
            vehicle_idx: Index of the vehicle
            en_idx: Index of the edge node
            offload_ratio: Offloading ratio o_n
            num_associated: Number of vehicles associated with this EN

        Returns:
            Offloading energy consumption in Joules
        """
        task = self.current_tasks[vehicle_idx]
        if task is None or offload_ratio == 0:
            return 0

        data_size, _ = task
        offload_portion = offload_ratio * data_size

        # Transmission time
        rate = self._compute_transmission_rate(vehicle_idx, en_idx, num_associated)
        rate = max(rate, 1e-6)
        tau_trans = offload_portion / rate

        # Transmission energy
        p_n = self.net_cfg['vehicle_transmit_power']
        energy = p_n * tau_trans

        return energy

    def _compute_uav_energy(self, velocity):
        """
        Compute UAV propulsion energy consumption

        Args:
            velocity: UAV flight velocity

        Returns:
            UAV energy consumption in Joules
        """
        P0 = self.net_cfg['P0']
        Pi = self.net_cfg['Pi']
        Utip = self.net_cfg['Utip']
        v0 = self.net_cfg['v0']
        d0 = self.net_cfg['d0']
        rho0 = self.net_cfg['rho0']
        s0 = self.net_cfg['s0']
        A0 = self.net_cfg['A0']
        tau = self.tau

        term1 = P0 * (1 + 3 * velocity**2 / Utip**2)
        inner = (1 + velocity**4 / (4 * v0**4))**0.5 - velocity**2 / (2 * v0**2)
        inner = max(inner, 0.0)  # ← CRITICAL numerical guard
        term2 = Pi * (inner ** 0.5)
        term3 = 0.5 * d0 * rho0 * s0 * A0 * velocity**3

        energy = tau * (term1 + term2 + term3)

        return energy

    def _compute_uav_computation_energy(self, en_idx, offload_ratio, data_size):
        """
        UAV computation energy
        Only applies when task is offloaded to UAV
        """
        if en_idx != self.num_rsus or offload_ratio == 0:
            return 0

        kappa_u = self.net_cfg['kappa_uav']
        F_u = self.net_cfg['uav_compute_capacity']
        X_c = self.net_cfg['cpu_cycles_per_bit']

        tau_comp = offload_ratio * X_c * data_size / F_u
        energy = kappa_u * (F_u ** 3) * tau_comp

        return energy

    def _compute_rental_price(self, vehicle_idx, en_idx, offload_ratio):
        """
        Compute rental price

        Args:
            vehicle_idx: Index of the vehicle
            en_idx: Index of the edge node
            offload_ratio: Offloading ratio o_n

        Returns:
            Rental price
        """
        task = self.current_tasks[vehicle_idx]
        if task is None or offload_ratio == 0:
            return 0

        data_size, _ = task
        X_c = self.net_cfg['cpu_cycles_per_bit']

        if en_idx < self.num_rsus:
            price_per_cycle = self.net_cfg['rsu_rental_price']
        else:
            price_per_cycle = self.net_cfg['uav_rental_price']

        rental_price = price_per_cycle * offload_ratio * X_c * data_size

        return rental_price

    #  TOPSIS Load Balancing (RSU-only, I2I model, UAV independent)

    def _is_en_overloaded(self, en_idx):
        if en_idx >= self.num_rsus:
            return False
        ratio = self.net_cfg.get('load_threshold_ratio', 0.7)
        cap = self.net_cfg['rsu_compute_capacity']
        X_c = self.net_cfg['cpu_cycles_per_bit']
        queue_time = X_c * self.en_queues[en_idx] / cap
        return queue_time > ratio * self.net_cfg['task_max_latency']

    def _topsis_select_server(self, overloaded_en, vehicle_idx):
        w_r = self.net_cfg.get('topsis_w_resource', 0.6)
        w_d = self.net_cfg.get('topsis_w_distance', 0.4)
        X_c = self.net_cfg['cpu_cycles_per_bit']
        ov_pos = np.array(self.rsu_positions[overloaded_en], dtype=float)
        cands = []
        for m in range(self.num_rsus):  # RSUs only, not UAV
            if m == overloaded_en: continue
            cap = self.net_cfg['rsu_compute_capacity']
            margin = max(cap - X_c * self.en_queues[m], 0)
            m_pos = np.array(self.rsu_positions[m], dtype=float)
            dist = np.linalg.norm(ov_pos - m_pos)
            cands.append((m, margin, dist))
        if not cands: return -1
        resources = np.array([c[1] for c in cands])
        distances = np.array([c[2] for c in cands])
        nr = (resources - resources.min()) / (resources.max() - resources.min() + 1e-10)
        nd = (distances.max() - distances) / (distances.max() - distances.min() + 1e-10)
        wr = w_r * nr; wd = w_d * nd
        d_plus = np.array([wr.max(), wd.max()])
        d_minus = np.array([wr.min(), wd.min()])
        best_score, best_en = -1, -1
        for i, (m, _, _) in enumerate(cands):
            pt = np.array([wr[i], wd[i]])
            e_plus = np.linalg.norm(pt - d_plus)
            e_minus = np.linalg.norm(pt - d_minus)
            score = e_minus / (e_plus + e_minus + 1e-10)
            if score > best_score:
                best_score = score; best_en = m
        if best_en >= 0 and not self._is_en_overloaded(best_en):
            return best_en
        return -1

    def _get_state(self):
        """
        Construct state vector

        Returns:
            State vector as numpy array
        """
        state = []

        # W(t): Vehicle and UAV positions (2(N+1) dimensions)
        for pos in self.vehicle_positions:
            state.extend(pos)
        state.extend(self.uav_position)

        # U(t): Task information (2N dimensions)

        for task in self.current_tasks:
            if task is not None:
                state.extend([task[0], task[1]])  # (D_n, T_max_n)
            else:
                state.extend([0, 0])

        # Y(t): Task intervals (N dimensions)
        state.extend(self.task_intervals)

        # Q(t): Queue backlogs (N + M + 1 dimensions)
        state.extend(self.vehicle_queues)
        state.extend(self.en_queues)

        # H(t): Channel gains (N(M+1) dimensions)
        channel_gains = self._compute_channel_gains()
        state.extend(channel_gains.flatten())

        # E_res_u: UAV residual energy (1 dimension)
        state.append(self.uav_residual_energy)

        # return np.array(state, dtype=np.float32)
        state = np.array(state, dtype=np.float32)
        state = self._normalize_state(state)
        return state


    def step(self, action):
        """
        Execute one time step

        Args:
            action: Action vector [v_u, theta_u, alpha_1, ..., alpha_N, p_1, ..., p_N] (if NOMA)
                    or [v_u, theta_u, alpha_1, ..., alpha_N] (if Base)

        Returns:
            next_state, reward, done, info
        """

        scaled_action = action * self.action_space.high

        v_u = scaled_action[0]
        theta_u = scaled_action[1]
        alphas = scaled_action[2 : 2 + self.num_vehicles]


        noma_on = self.net_cfg.get('noma_enabled', False)

        if noma_on:
            powers = scaled_action[2 + self.num_vehicles : ]  # Extract dynamically allocated powers
        else:
            # Base model uses static transmit power from config
            powers = np.full(self.num_vehicles, self.net_cfg['vehicle_transmit_power'])

        # Update UAV position (equations 1-2)
        self.uav_position[0] += self.tau * v_u * np.cos(theta_u)
        self.uav_position[1] += self.tau * v_u * np.sin(theta_u)

        # Keep UAV within map bounds
        area_width, area_height = self.map_cfg['area']
        self.uav_position[0] = np.clip(self.uav_position[0], 0, area_width)
        self.uav_position[1] = np.clip(self.uav_position[1], 0, area_height)

        # Update vehicle positions
        for n in range(self.num_vehicles):
            self.vehicle_positions[n, 0] += self.tau * self.vehicle_velocities[n] * np.cos(self.vehicle_angles[n])
            self.vehicle_positions[n, 1] += self.tau * self.vehicle_velocities[n] * np.sin(self.vehicle_angles[n])

            # Wrap around or reflect at boundaries
            self.vehicle_positions[n, 0] = np.clip(self.vehicle_positions[n, 0], 0, area_width)
            self.vehicle_positions[n, 1] = np.clip(self.vehicle_positions[n, 1], 0, area_height)

        # Decode user association from alpha (equation 30-31)
        associations = np.zeros((self.num_vehicles, self.num_ens), dtype=int)
        offload_ratios = np.zeros(self.num_vehicles)

        for n in range(self.num_vehicles):
            alpha_n = alphas[n]

            if alpha_n <= 1/3:
                # Local computation
                associations[n, :] = 0
                offload_ratios[n] = 0
            elif alpha_n <= 2/3:
                # Offload to RSU
                vehicle_pos = self.vehicle_positions[n]
                rsu_radius = self.net_cfg['rsu_coverage_radius']

                # find all RSUs in coverage
                distances = [
                    np.linalg.norm(vehicle_pos - rsu_pos)
                    for rsu_pos in self.rsu_positions
                ]

                in_range = [i for i, d in enumerate(distances) if d <= rsu_radius]

                if in_range:
                    best_rsu = min(in_range, key=lambda i: distances[i])
                else:
                    best_rsu = np.argmin(distances)

                associations[n, best_rsu] = 1
                offload_ratios[n] = np.clip(3 * alpha_n - 1, 0, 1)
            else:
                uav_idx = self.num_rsus  # UAV is last EN
                associations[n, uav_idx] = 1
                offload_ratios[n] = np.clip(3 * alpha_n - 2, 0, 1)

        # TOPSIS Load Balancing (I2I model: vehicle->nearest RSU->wired->migrated RSU)
        topsis_on = self.net_cfg.get('topsis_enabled', False)
        migrations = 0
        migrated_to = {}

        if topsis_on:
            for n in range(self.num_vehicles):
                if associations[n].sum() == 0: continue
                en_idx = int(np.argmax(associations[n]))
                if self._is_en_overloaded(en_idx):
                    alt = self._topsis_select_server(en_idx, n)
                    if alt >= 0:
                        migrated_to[n] = alt
                        migrations += 1

        en_associations = associations.sum(axis=0)

        # Compute costs for each vehicle
        total_cost = 0
        penalty = 0
        starvation_penalty = 0

        # Trackers for specific physical metrics
        total_aoi = 0
        total_energy = 0
        total_per = 0
        total_price = 0
        offloaded_tasks = 0

        # --- Trackers for WEIGHTED cost components ---
        weighted_aoi_cost_total = 0
        weighted_energy_cost_total = 0
        weighted_price_cost_total = 0
        weighted_per_cost_total = 0

        min_required_rate = 1e6

        # 2. Pre-compute rates based on the active scheme
        rates = np.zeros(self.num_vehicles)
        pers = np.zeros(self.num_vehicles)

        # Calculate ALL gains exactly once here
        current_gains = self._compute_channel_gains()

        if noma_on:
            rates, pers, sic_violations = self._compute_noma_rates_and_per(associations, powers)
            penalty += (sic_violations * 5.0)  # Punish agent for breaking NOMA rules
        else:
            for n in range(self.num_vehicles):
                en_idx = np.argmax(associations[n]) if associations[n].sum() > 0 else -1
                if en_idx >= 0:
                    num_assoc = en_associations[en_idx]
                    # Extract the specific gain and pass it in
                    h_nm = current_gains[n, en_idx]
                    rates[n], pers[n] = self._compute_transmission_rate(n, en_idx, num_assoc, h_nm)



        # 3. Cost and Environment Update Loop
        for n in range(self.num_vehicles):
            task = self.current_tasks[n]
            if task is None:
                continue

            # Find associated EN
            en_idx = np.argmax(associations[n]) if associations[n].sum() > 0 else -1
            offload_ratio = offload_ratios[n]
            data_size, max_latency = task

            # Compute local latencies and energies
            tau_local = self._compute_local_latency(n, offload_ratio)
            energy_local = self._compute_local_energy(n, offload_ratio)

            tau_offload = 0
            energy_offload = 0
            per_n = 0
            rental_price = 0

            # Offloading calculations
            if en_idx >= 0 and offload_ratio > 0:
                offload_portion = offload_ratio * data_size
                rate = max(rates[n], 1e-6)

                # Check for Starvation
                if rate < min_required_rate:
                    starvation_penalty += 5.0

                # Transmission time using the calculated rate
                tau_trans = offload_portion / rate
                tau_trans = min(tau_trans, 10.0)

                # Queue and compute at migrated RSU (if TOPSIS moved it)
                compute_en = migrated_to.get(n, en_idx) if topsis_on else en_idx
                F_m = self.net_cfg['rsu_compute_capacity'] if compute_en < self.num_rsus else self.net_cfg['uav_compute_capacity']
                X_c = self.net_cfg['cpu_cycles_per_bit']
                Q_m = self.en_queues[compute_en]

                tau_queue = X_c * Q_m / F_m
                tau_comp = offload_ratio * X_c * data_size / F_m

                tau_offload = tau_trans + tau_queue + tau_comp

                # Energy calculation
                energy_offload = powers[n] * tau_trans
                per_n = pers[n]
                rental_price = self._compute_rental_price(n, en_idx, offload_ratio)

            # Total execution time and energy
            tau_total = max(tau_local, tau_offload)
            energy_total = energy_local + energy_offload

            # Update AoI (equation 24)
            aoi_cost = tau_total + self.task_intervals[n]

            # Update Physical Trackers
            total_aoi += aoi_cost
            total_energy += energy_total
            total_price += rental_price
            if en_idx >= 0 and offload_ratio > 0:
                total_per += per_n
                offloaded_tasks += 1

            if en_idx == self.num_rsus and offload_ratio > 0:
                uav_comp_energy = self._compute_uav_computation_energy(
                    en_idx, offload_ratio, data_size
                )
                self.uav_residual_energy -= uav_comp_energy

            # Integrate Costs
            gamma_A = self.net_cfg['gamma_A']
            gamma_E = self.net_cfg['gamma_E']
            gamma_P = self.net_cfg['gamma_P']
            gamma_R = self.net_cfg.get('gamma_R', 0.0)

            # Calculate individual weighted components
            w_aoi = gamma_A * aoi_cost
            w_energy = gamma_E * energy_total
            w_price = gamma_P * rental_price
            w_per = gamma_R * per_n

            # Add to trackers
            weighted_aoi_cost_total += w_aoi
            weighted_energy_cost_total += w_energy
            weighted_price_cost_total += w_price
            weighted_per_cost_total += w_per

            vehicle_cost = w_aoi + w_energy + w_price + w_per
            total_cost += vehicle_cost

            if tau_total > max_latency:
                excess_time = tau_total - max_latency
                penalty += self.net_cfg['c2'] * (1.0 + excess_time)

            # Update queues
            F_n = self.net_cfg['vehicle_compute_capacity']
            X_c = self.net_cfg['cpu_cycles_per_bit']
            D_comp_n = min(self.tau * F_n / X_c, self.vehicle_queues[n])
            self.vehicle_queues[n] = max(self.vehicle_queues[n] - D_comp_n, 0) + (1 - offload_ratio) * data_size

            if en_idx >= 0 and offload_ratio > 0:
                compute_en = migrated_to.get(n, en_idx) if topsis_on else en_idx
                if compute_en < self.num_rsus:
                    F_m = self.net_cfg['rsu_compute_capacity']
                else:
                    F_m = self.net_cfg['uav_compute_capacity']

                D_comp_m = min(self.tau * F_m / X_c, self.en_queues[compute_en])
                self.en_queues[compute_en] = max(self.en_queues[compute_en] - D_comp_m, 0) + offload_ratio * data_size

        for m in range(self.num_rsus):
            max_assoc = self.net_cfg['rsu_max_associations']
            if en_associations[m] > max_assoc:
                # Multiply c1 by the number of vehicles exceeding the limit
                overload_count = en_associations[m] - max_assoc
                penalty += self.net_cfg['c1'] * overload_count

        if en_associations[self.num_rsus] > self.net_cfg['uav_max_associations']:
            overload_count = en_associations[self.num_rsus] - self.net_cfg['uav_max_associations']
            penalty += self.net_cfg['c1'] * overload_count

        # Update UAV energy
        uav_energy_consumed = self._compute_uav_energy(v_u)
        self.uav_residual_energy -= uav_energy_consumed

        if self.uav_residual_energy < 0:
            penalty += self.net_cfg['c3']

        # Compute reward
        reward = -(total_cost + penalty + starvation_penalty) * 0.1

        # Update time and tasks
        self.task_intervals += self.tau
        self.current_tasks, arrivals = self._generate_tasks()
        self.task_intervals[arrivals == 1] = 0

        self.current_step += 1
        done = self.current_step >= self.td3_cfg['max_steps']

        info = {
            'total_cost': total_cost,
            'penalty': penalty,
            'starvation_penalty': starvation_penalty,
            'uav_energy': self.uav_residual_energy,
            'avg_aoi': total_aoi / self.num_vehicles,
            'avg_energy': total_energy / self.num_vehicles,
            'avg_price': total_price / self.num_vehicles,
            'avg_per': total_per / max(offloaded_tasks, 1),
            'base_cost': (self.net_cfg['gamma_A'] * total_aoi +
                          self.net_cfg['gamma_E'] * total_energy +
                          self.net_cfg['gamma_P'] * total_price),
            # --- Export weighted cost components ---
            'cost_comp_aoi': weighted_aoi_cost_total,
            'cost_comp_energy': weighted_energy_cost_total,
            'cost_comp_price': weighted_price_cost_total,
            'cost_comp_per': weighted_per_cost_total
        }

        return self._get_state(), reward, done, info

# **Networks.py**

In [ ]:
"""
TD3 Networks: Actor and Critic networks
Implements the neural network architecture
"""

import torch
import torch.nn as nn
import torch.nn.functional as F


class Actor(nn.Module):
    """
    Actor network for TD3
    Architecture: Input -> [600, 400] -> Output
    Activation: ReLU for hidden layers, Tanh for output
    """

    def __init__(self, state_dim, action_dim, max_action):
        super(Actor, self).__init__()

        # self.max_action = max_action
        self.register_buffer(
            "max_action",
            torch.tensor(max_action, dtype=torch.float32)
        )


        self.fc1 = nn.Linear(state_dim, 600)
        self.fc2 = nn.Linear(600, 400)
        self.fc3 = nn.Linear(400, action_dim)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize network weights"""
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, state):
        x = F.relu(self.fc1(state))
        x = F.relu(self.fc2(x))
        x = torch.tanh(self.fc3(x))
        return (x + 1) / 2

class Critic(nn.Module):
    """
    Critic network for TD3
    Architecture: Input (state+action) -> [600, 400] -> Output (Q-value)
    Activation: ReLU for hidden layers
    """

    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()

        # Q1 architecture
        self.fc1 = nn.Linear(state_dim + action_dim, 600)
        self.fc2 = nn.Linear(600, 400)
        self.fc3 = nn.Linear(400, 1)

        # Q2 architecture (for twin critics)
        self.fc4 = nn.Linear(state_dim + action_dim, 600)
        self.fc5 = nn.Linear(600, 400)
        self.fc6 = nn.Linear(400, 1)

        # Initialize weights
        self._init_weights()

    def _init_weights(self):
        """Initialize network weights"""
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)
        nn.init.xavier_uniform_(self.fc4.weight)
        nn.init.xavier_uniform_(self.fc5.weight)
        nn.init.xavier_uniform_(self.fc6.weight)

    def forward(self, state, action):
        """
        Forward pass for both Q networks

        Args:
            state: State tensor
            action: Action tensor

        Returns:
            Q1 and Q2 values
        """
        sa = torch.cat([state, action], dim=1)

        # Q1 forward pass
        q1 = F.relu(self.fc1(sa))
        q1 = F.relu(self.fc2(q1))
        q1 = self.fc3(q1)

        # Q2 forward pass
        q2 = F.relu(self.fc4(sa))
        q2 = F.relu(self.fc5(q2))
        q2 = self.fc6(q2)

        return q1, q2

    def Q1(self, state, action):
        """
        Return only Q1 value (used for actor update)

        Args:
            state: State tensor
            action: Action tensor

        Returns:
            Q1 value
        """
        sa = torch.cat([state, action], dim=1)

        q1 = F.relu(self.fc1(sa))
        q1 = F.relu(self.fc2(q1))
        q1 = self.fc3(q1)

        return q1


class ReplayBuffer:
    """
    Replay buffer for storing and sampling transitions
    """

    def __init__(self, state_dim, action_dim, max_size=1000000):
        self.max_size = max_size
        self.ptr = 0
        self.size = 0

        self.state = torch.zeros((max_size, state_dim))
        self.action = torch.zeros((max_size, action_dim))
        self.next_state = torch.zeros((max_size, state_dim))
        self.reward = torch.zeros((max_size, 1))
        self.done = torch.zeros((max_size, 1))

        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    def add(self, state, action, next_state, reward, done):
        """
        Add a transition to the buffer

        Args:
            state: Current state
            action: Action taken
            next_state: Next state
            reward: Reward received
            done: Done flag
        """
        self.state[self.ptr] = torch.FloatTensor(state)
        self.action[self.ptr] = torch.FloatTensor(action)
        self.next_state[self.ptr] = torch.FloatTensor(next_state)
        self.reward[self.ptr] = torch.FloatTensor([reward])
        self.done[self.ptr] = torch.FloatTensor([done])

        self.ptr = (self.ptr + 1) % self.max_size
        self.size = min(self.size + 1, self.max_size)

    def sample(self, batch_size):
        """
        Sample a batch of transitions

        Args:
            batch_size: Size of the batch

        Returns:
            Batch of (state, action, next_state, reward, done)
        """
        ind = torch.randint(0, self.size, size=(batch_size,))

        return (
            self.state[ind].to(self.device),
            self.action[ind].to(self.device),
            self.next_state[ind].to(self.device),
            self.reward[ind].to(self.device),
            self.done[ind].to(self.device)
        )

# **TD3Agent.py**

In [ ]:
"""
TD3 Agent Implementation
"""

import torch
import torch.nn.functional as F
# from networks import Actor, Critic, ReplayBuffer


class TD3Agent:
    """
    Twin Delayed Deep Deterministic Policy Gradient (TD3) Agent
    Implements Algorithm
    """

    def __init__(self, state_dim, action_dim, max_action, config):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        self.config = config
        self.state_dim = state_dim
        self.action_dim = action_dim
        # self.max_action = max_action
        self.max_action = torch.tensor(max_action, dtype=torch.float32, device=self.device)


        # Hyperparameters
        self.actor_lr = config['actor_lr']
        self.critic_lr = config['critic_lr']
        self.discount = config['discount_factor']
        self.tau = config['tau']
        self.policy_noise = config['policy_noise']
        self.noise_clip = config['noise_clip']
        self.policy_delay = config['policy_delay']
        self.batch_size = config['batch_size']
        self.exploration_noise = 0.15
        self.noise_decay_rate = 0.999
        self.noise_min = 0.02

        # Initialize actor networks
        self.actor = Actor(state_dim, action_dim, max_action).to(self.device)
        self.actor_target = Actor(state_dim, action_dim, max_action).to(self.device)
        self.actor_target.load_state_dict(self.actor.state_dict())
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=self.actor_lr)

        # Initialize critic networks
        self.critic = Critic(state_dim, action_dim).to(self.device)
        self.critic_target = Critic(state_dim, action_dim).to(self.device)
        self.critic_target.load_state_dict(self.critic.state_dict())
        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=self.critic_lr)

        # Initialize replay buffer
        self.replay_buffer = ReplayBuffer(state_dim, action_dim, config['buffer_size'])

        # Training step counter
        self.total_it = 0

    def select_action(self, state, add_noise=True, noise_scale=0.1):
        """
        Select action using the actor network

        Args:
            state: Current state
            add_noise: Whether to add exploration noise
            noise_scale: Scale of the exploration noise

        Returns:
            Action array
        """

        if isinstance(state, np.ndarray):
            state = torch.FloatTensor(state.reshape(1, -1)).to(self.device)
        else:
            state = state.reshape(1, -1)
        action = self.actor(state).detach().cpu().numpy().flatten()


        if add_noise:
            max_a = self.max_action.cpu().numpy()
            noise = np.random.normal(0, noise_scale, size=self.action_dim)
            action = action + noise
            action = np.clip(action, 0, 1)

        return action

    def train(self):
        """
        Train the agent for one step
        Implements Lines 10-18 of Algorithm 1

        Returns:
            Dictionary containing training metrics
        """
        if self.replay_buffer.size < self.batch_size:
          return {'critic_loss': None, 'actor_loss': None}
        self.total_it += 1

        # Sample mini-batch from replay buffer
        state, action, next_state, reward, done = self.replay_buffer.sample(self.batch_size)


        with torch.no_grad():
            noise = torch.randn_like(action) * self.policy_noise
            noise = torch.clamp(noise, -self.noise_clip, self.noise_clip)
            next_action = self.actor_target(next_state) + noise
            next_action = torch.clamp(next_action, min=0.0, max=1.0)


            # Compute target Q-value
            target_Q1, target_Q2 = self.critic_target(next_state, next_action)
            target_Q = torch.min(target_Q1, target_Q2)
            target_Q = reward + (1 - done) * self.discount * target_Q

        # Get current Q estimates
        current_Q1, current_Q2 = self.critic(state, action)

        # Compute critic loss
        critic_loss = F.smooth_l1_loss(current_Q1, target_Q) + F.smooth_l1_loss(current_Q2, target_Q)

        # Optimize the critic
        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic.parameters(), 10.0)
        self.critic_optimizer.step()

        # Delayed policy updates
        actor_loss = None
        if self.total_it % self.policy_delay == 0:
            # Compute actor loss
            actor_loss = -self.critic.Q1(state, self.actor(state)).mean()

            # Optimize the actor
            self.actor_optimizer.zero_grad()
            actor_loss.backward()
            self.actor_optimizer.step()

            # Update target networks
            for param, target_param in zip(self.critic.parameters(), self.critic_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

            for param, target_param in zip(self.actor.parameters(), self.actor_target.parameters()):
                target_param.data.copy_(self.tau * param.data + (1 - self.tau) * target_param.data)

        metrics = {
            'critic_loss': critic_loss.item(),
            'actor_loss': actor_loss.item() if actor_loss is not None else None,
        }

        return metrics

    def decay_noise(self):
        self.exploration_noise = max(self.noise_min, self.exploration_noise * self.noise_decay_rate)

    def save(self, filename):
        """
        Save model parameters

        Args:
            filename: Path to save the model
        """
        torch.save({
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict(),
            'actor_target': self.actor_target.state_dict(),
            'critic_target': self.critic_target.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict(),
        }, filename)

    def load(self, filename):
        """
        Load model parameters

        Args:
            filename: Path to load the model from
        """
        checkpoint = torch.load(filename)
        self.actor.load_state_dict(checkpoint['actor'])
        self.critic.load_state_dict(checkpoint['critic'])
        self.actor_target.load_state_dict(checkpoint['actor_target'])
        self.critic_target.load_state_dict(checkpoint['critic_target'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer'])

    def save_checkpoint(self, filename, episode, metrics=None):
        """
        Save full training checkpoint for resuming later.
        Includes model weights, optimizer states, replay buffer, and metrics.
        """
        checkpoint = {
            'actor': self.actor.state_dict(),
            'critic': self.critic.state_dict(),
            'actor_target': self.actor_target.state_dict(),
            'critic_target': self.critic_target.state_dict(),
            'actor_optimizer': self.actor_optimizer.state_dict(),
            'critic_optimizer': self.critic_optimizer.state_dict(),
            'episode': episode,
            'total_it': self.total_it,
            'exploration_noise': self.exploration_noise,
            'replay_buffer': {
                'state': self.replay_buffer.state[:self.replay_buffer.size],
                'action': self.replay_buffer.action[:self.replay_buffer.size],
                'next_state': self.replay_buffer.next_state[:self.replay_buffer.size],
                'reward': self.replay_buffer.reward[:self.replay_buffer.size],
                'done': self.replay_buffer.done[:self.replay_buffer.size],
                'size': self.replay_buffer.size,
                'ptr': self.replay_buffer.ptr,
            },
        }
        if metrics is not None:
            checkpoint['metrics'] = metrics
        torch.save(checkpoint, filename)
        print(f"  Checkpoint saved: episode {episode} \u2192 {filename}")

    def load_checkpoint(self, filename):
        """
        Load full training checkpoint. Returns (episode, metrics) to resume.
        """
        checkpoint = torch.load(filename, map_location='cpu',weights_only=False)
        self.actor.load_state_dict(checkpoint['actor'])
        self.critic.load_state_dict(checkpoint['critic'])
        self.actor_target.load_state_dict(checkpoint['actor_target'])
        self.critic_target.load_state_dict(checkpoint['critic_target'])
        self.actor_optimizer.load_state_dict(checkpoint['actor_optimizer'])
        self.critic_optimizer.load_state_dict(checkpoint['critic_optimizer'])
        self.total_it = checkpoint.get('total_it', 0)
        self.exploration_noise = checkpoint.get('exploration_noise', 0.15)

        # Restore replay buffer
        buf = checkpoint.get('replay_buffer')
        if buf is not None:
            size = buf['size']
            self.replay_buffer.state[:size] = buf['state']
            self.replay_buffer.action[:size] = buf['action']
            self.replay_buffer.next_state[:size] = buf['next_state']
            self.replay_buffer.reward[:size] = buf['reward']
            self.replay_buffer.done[:size] = buf['done']
            self.replay_buffer.size = size
            self.replay_buffer.ptr = buf['ptr']

        episode = checkpoint.get('episode', 0)
        metrics = checkpoint.get('metrics', None)
        print(f"  Checkpoint loaded: episode {episode}, buffer size {self.replay_buffer.size}")
        return episode, metrics


In [ ]:
"""
Main Training Script Algorithm
"""

import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm
import json
import os


def train_drl_tcoa(config, save_dir=DRIVE_SAVE_DIR, resume_from=None,
                   checkpoint_every=100):
    """
    Train the DRL-TCOA algorithm with checkpoint support.
    """
    # Create save directory
    os.makedirs(save_dir, exist_ok=True)

    # Initialize environment
    env = UAVAssistedVECEnv(config)

    # Get dimensions
    state_dim = env.observation_space.shape[0]
    action_dim = env.action_space.shape[0]
    max_action = env.action_space.high

    print(f"State dimension: {state_dim}")
    print(f"Action dimension: {action_dim}")
    print(f"Max action shape: {max_action.shape}")

    # Initialize TD3 agent
    agent = TD3Agent(state_dim, action_dim, max_action, config['td3'])

    # Training parameters
    max_episodes = config['td3']['max_episodes']
    max_steps = config['td3']['max_steps']
    batch_size = config['td3']['batch_size']

    # Metrics storage
    episode_rewards = []
    episode_costs = []
    episode_penalties = []
    critic_losses = []
    actor_losses = []
    episode_aois = []
    episode_energies = []
    episode_prices = []
    episode_pers = []
    episode_base_costs = []

    # --- Storage for cost components ---
    episode_cost_comp_aoi = []
    episode_cost_comp_energy = []
    episode_cost_comp_price = []
    episode_cost_comp_per = []

    warmup_steps = 5000
    start_episode = 0

    # ─── Resume from checkpoint if provided ──────────────────────────────────
    if resume_from is not None and os.path.exists(resume_from):
        print(f"\nResuming from checkpoint: {resume_from}")
        start_episode, saved_metrics = agent.load_checkpoint(resume_from)
        if saved_metrics is not None:
            episode_rewards = saved_metrics.get('episode_rewards', [])
            episode_costs = saved_metrics.get('episode_costs', [])
            episode_penalties = saved_metrics.get('episode_penalties', [])
            critic_losses = saved_metrics.get('critic_losses', [])
            actor_losses = saved_metrics.get('actor_losses', [])
            episode_aois = saved_metrics.get('episode_aois', [])
            episode_energies = saved_metrics.get('episode_energies', [])
            episode_pers = saved_metrics.get('episode_pers', [])
            episode_base_costs = saved_metrics.get('episode_base_costs', [])
            episode_cost_comp_aoi = saved_metrics.get('cost_comp_aoi', [])
            episode_cost_comp_energy = saved_metrics.get('cost_comp_energy', [])
            episode_cost_comp_price = saved_metrics.get('cost_comp_price', [])
            episode_cost_comp_per = saved_metrics.get('cost_comp_per', [])
        print(f"  Resuming from episode {start_episode}/{max_episodes}")
        print(f"  Existing metrics: {len(episode_rewards)} episodes")

    # Training loop
    for episode in tqdm(range(start_episode, max_episodes), desc="Training",
                        initial=start_episode, total=max_episodes):

        state = env.reset()
        episode_reward = 0
        episode_cost = 0
        episode_penalty = 0

        # Reset episode trackers
        episode_aoi = 0
        episode_energy = 0
        episode_price = 0
        episode_per = 0
        episode_base_cost = 0

        # Breakdown trackers
        ep_comp_aoi = 0
        ep_comp_energy = 0
        ep_comp_price = 0
        ep_comp_per = 0

        num_steps = 0

        for step in range(max_steps):
            if agent.replay_buffer.size < warmup_steps:
                # Sample uniform random actions strictly in [0, 1] range
                action = np.random.uniform(0, 1, size=action_dim)
            else:
                action = agent.select_action(state, add_noise=True, noise_scale=agent.exploration_noise)

            next_state, reward, done, info = env.step(action)
            agent.replay_buffer.add(state, action, next_state, reward, done)

            # Update metrics
            raw_reward = -(info['total_cost'] + info['penalty'] + info.get('starvation_penalty', 0))
            episode_reward += raw_reward
            episode_cost += info['total_cost']
            episode_penalty += info['penalty']

            episode_aoi += info['avg_aoi']
            episode_energy += info['avg_energy']
            episode_price += info['avg_price']
            episode_per += info['avg_per']
            episode_base_cost += info['base_cost']

            # Component breakdown
            ep_comp_aoi += info['cost_comp_aoi']
            ep_comp_energy += info['cost_comp_energy']
            ep_comp_price += info['cost_comp_price']
            ep_comp_per += info['cost_comp_per']

            # Train agent if buffer is full
            if agent.replay_buffer.size >= batch_size:
                metrics = agent.train()
                if metrics['critic_loss'] is not None:
                    critic_losses.append(metrics['critic_loss'])
                if metrics['actor_loss'] is not None:
                    actor_losses.append(metrics['actor_loss'])

            state = next_state
            if done:
                num_steps = step + 1
                break

        # Store episode metrics
        episode_rewards.append(episode_reward/num_steps)
        episode_costs.append(episode_cost/num_steps)
        episode_penalties.append(episode_penalty/num_steps)

        episode_aois.append(episode_aoi/num_steps)
        episode_energies.append(episode_energy/num_steps)
        episode_prices.append(episode_price/num_steps)
        episode_pers.append(episode_per/num_steps)
        episode_base_costs.append(episode_base_cost/num_steps)

        episode_cost_comp_aoi.append(ep_comp_aoi/num_steps)
        episode_cost_comp_energy.append(ep_comp_energy/num_steps)
        episode_cost_comp_price.append(ep_comp_price/num_steps)
        episode_cost_comp_per.append(ep_comp_per/num_steps)

        agent.decay_noise()

        if (episode + 1) % 10 == 0:
            avg_reward = np.mean(episode_rewards[-10:])
            avg_cost = np.mean(episode_costs[-10:])
            print(f"\nEpisode {episode + 1}/{max_episodes}")
            print(f"  Avg Reward (last 10): {avg_reward:.2f}")
            print(f"  Avg Cost (last 10): {avg_cost:.2f}")
            if len(critic_losses) > 0:
                print(f"  Avg Critic Loss: {np.mean(critic_losses[-100:]):.4f}")
            if len(actor_losses) > 0:
                print(f"  Avg Actor Loss: {np.mean(actor_losses[-100:]):.4f}")

        # ─── Save checkpoint periodically ──────────────────────────────────────
        if (episode + 1) % checkpoint_every == 0:
            ckpt_metrics = {
                'episode_rewards': episode_rewards,
                'episode_costs': episode_costs,
                'episode_penalties': episode_penalties,
                'critic_losses': critic_losses,
                'actor_losses': actor_losses,
                'episode_aois': episode_aois,
                'episode_energies': episode_energies,
                'episode_prices': episode_prices,
                'episode_pers': episode_pers,
                'episode_base_costs': episode_base_costs,
                'cost_comp_aoi': episode_cost_comp_aoi,
                'cost_comp_energy': episode_cost_comp_energy,
                'cost_comp_price': episode_cost_comp_price,
                'cost_comp_per': episode_cost_comp_per,
            }
            # ckpt_path = os.path.join(save_dir, f'checkpoint_ep{episode+1}.pth')
            # agent.save_checkpoint(ckpt_path, episode + 1, ckpt_metrics)
            latest_path = os.path.join(save_dir, 'checkpoint_latest.pth')
            agent.save_checkpoint(latest_path, episode + 1, ckpt_metrics)

    # Save model
    model_path = os.path.join(save_dir, 'td3_model.pth')
    agent.save(model_path)
    print(f"\nModel saved to {model_path}")

    # Save metrics
    metrics = {
        'episode_rewards': episode_rewards,
        'episode_costs': episode_costs,
        'episode_penalties': episode_penalties,
        'critic_losses': critic_losses,
        'actor_losses': actor_losses,
        'episode_aois': episode_aois,
        'episode_energies': episode_energies,
        'episode_prices': episode_prices,
        'episode_pers': episode_pers,
        'episode_base_costs': episode_base_costs,
        'cost_comp_aoi': episode_cost_comp_aoi,
        'cost_comp_energy': episode_cost_comp_energy,
        'cost_comp_price': episode_cost_comp_price,
        'cost_comp_per': episode_cost_comp_per,
    }

    metrics_path = os.path.join(save_dir, 'training_metrics.json')
    with open(metrics_path, 'w') as f:
        json_metrics = {k: [float(v) for v in vals] for k, vals in metrics.items()}
        json.dump(json_metrics, f, indent=2)
    print(f"Metrics saved to {metrics_path}")

    return metrics, agent


def evaluate_agent(agent, config, num_episodes=10, seed=42):
    """
    Evaluate trained agent on a deterministic set of tasks for fair comparison.
    """
    env = UAVAssistedVECEnv(config)

    # Lock the random seeds for reproducibility
    np.random.seed(seed)
    torch.manual_seed(seed)

    episode_rewards = []
    episode_costs = []
    episode_penalties = []
    episode_aois = []
    episode_energies = []
    episode_prices = []
    episode_pers = []
    episode_base_costs = []

    eval_comp_aoi = []
    eval_comp_energy = []
    eval_comp_price = []
    eval_comp_per = []

    for episode in range(num_episodes):
        state = env.reset()
        episode_reward = 0
        episode_cost = 0
        episode_penalty = 0
        episode_aoi = 0
        episode_energy = 0
        episode_price = 0
        episode_per = 0
        episode_base_cost = 0

        ep_comp_aoi = 0
        ep_comp_energy = 0
        ep_comp_price = 0
        ep_comp_per = 0

        num_steps = 0

        for step in range(config['td3']['max_steps']):
            action = agent.select_action(state, add_noise=False)
            next_state, reward, done, info = env.step(action)

            raw_reward = -(info['total_cost'] + info['penalty'] + info.get('starvation_penalty', 0))
            episode_reward += raw_reward
            episode_cost += info['total_cost']
            episode_penalty += info['penalty']

            episode_aoi += info['avg_aoi']
            episode_energy += info['avg_energy']
            episode_price += info['avg_price']
            episode_per += info['avg_per']
            episode_base_cost += info['base_cost']

            ep_comp_aoi += info['cost_comp_aoi']
            ep_comp_energy += info['cost_comp_energy']
            ep_comp_price += info['cost_comp_price']
            ep_comp_per += info['cost_comp_per']

            state = next_state
            if done:
                num_steps = step+1
                break

        episode_rewards.append(episode_reward/num_steps)
        episode_costs.append(episode_cost/num_steps)
        episode_penalties.append(episode_penalty/num_steps)

        episode_aois.append(episode_aoi/num_steps)
        episode_energies.append(episode_energy/num_steps)
        episode_prices.append(episode_price/num_steps)
        episode_pers.append(episode_per/num_steps)
        episode_base_costs.append(episode_base_cost/num_steps)

        eval_comp_aoi.append(ep_comp_aoi/num_steps)
        eval_comp_energy.append(ep_comp_energy/num_steps)
        eval_comp_price.append(ep_comp_price/num_steps)
        eval_comp_per.append(ep_comp_per/num_steps)

    np.random.seed(None)

    results = {
        'mean_reward': np.mean(episode_rewards),
        'std_reward': np.std(episode_rewards),
        'mean_cost': np.mean(episode_costs),
        'std_cost': np.std(episode_costs),
        'mean_penalty': np.mean(episode_penalties),
        'std_penalty': np.std(episode_penalties),

        'mean_aoi': np.mean(episode_aois),
        'mean_energy': np.mean(episode_energies),
        'mean_price': np.mean(episode_prices),
        'mean_per': np.mean(episode_pers),
        'mean_base_cost': np.mean(episode_base_costs),

        # New evaluation averages for stacked bar chart
        'mean_comp_aoi': np.mean(eval_comp_aoi),
        'mean_comp_energy': np.mean(eval_comp_energy),
        'mean_comp_price': np.mean(eval_comp_price),
        'mean_comp_per': np.mean(eval_comp_per),
    }

    return results

def main():
    """Main training function for single-run VEC optimization"""

    # =========================================================================
    # 1. SELECT YOUR EXPERIMENT HERE
    # =========================================================================
    EXPERIMENT_NAME = 'NOMA_TD3_TOPSIS'  # Folders will be named this
    NOMA_ENABLED = True                  # True for NOMA, False for Base TD3
    TOPSIS_ENABLED = True                # True for TOPSIS load balancing
    # =========================================================================

    scenario = 'normal'
    map_name = 'map1'
    num_vehicles = 80

    print("\n" + "=" * 80)
    print(f"Starting VEC Training: {EXPERIMENT_NAME}")
    print("=" * 80)

    # 2. Create base configuration
    config = get_config(
        scenario=scenario,
        map_name=map_name,
        num_vehicles=num_vehicles
    )

    # 3. Inject algorithm-specific flags
    config['network']['noma_enabled'] = NOMA_ENABLED
    config['network']['topsis_enabled'] = TOPSIS_ENABLED

    # 4. Create experiment save directory
    drive_save_dir = (
        os.path.join(DRIVE_SAVE_DIR, "MAP1")
        if map_name == "map1"
        else os.path.join(DRIVE_SAVE_DIR, "MAP2")
    )

    save_dir = os.path.join(drive_save_dir, EXPERIMENT_NAME)
    os.makedirs(save_dir, exist_ok=True)

    print(f"  Scenario:       {scenario}")
    print(f"  Map:            {map_name}")
    print(f"  Vehicles:       {config['num_vehicles']}")
    print(f"  NOMA Enabled:   {NOMA_ENABLED}")
    print(f"  TOPSIS Enabled: {TOPSIS_ENABLED}")
    print(f"  Save Directory: {save_dir}")
    print("-" * 80)

    # 5. Auto-resume from checkpoint if it exists
    ckpt_path = os.path.join(save_dir, 'checkpoint_latest.pth')
    resume = ckpt_path if os.path.exists(ckpt_path) else None

    # 6. Train Agent
    print("\nStarting training...")
    metrics, agent = train_drl_tcoa(
        config,
        save_dir=save_dir,
        resume_from=resume,
        checkpoint_every=100
    )

    # 7. Evaluate Agent
    print("\nEvaluating trained agent...")
    eval_results = evaluate_agent(agent, config, num_episodes=10)

    print("\nEvaluation Results:")
    for key, value in eval_results.items():
        print(f"  {key}: {value:.4f}")

    # 8. Save evaluation results
    eval_path = os.path.join(save_dir, 'evaluation_results.json')
    with open(eval_path, 'w') as f:
        json.dump(eval_results, f, indent=2)

    print(f"\nSaved evaluation results to {eval_path}")
    print("=" * 80)
    print(f"Experiment {EXPERIMENT_NAME} completed successfully.")
    print("=" * 80)

if __name__ == "__main__":
    main()
